# Session – 11

# Advanced Pandas Data Engineering & Intelligent Data Access

In [2]:
import numpy as np
import pandas as pd

# Task 1: The Genetic Archivist (Schema Discovery & Column Isolation)

In [3]:
# Load dataset

df = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv")

In [4]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
print(df.columns)

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')


In [6]:
print(df.index)

RangeIndex(start=0, stop=768, step=1)


In [7]:
print(df.values)

[[  6.    148.     72.    ...   0.627  50.      1.   ]
 [  1.     85.     66.    ...   0.351  31.      0.   ]
 [  8.    183.     64.    ...   0.672  32.      1.   ]
 ...
 [  5.    121.     72.    ...   0.245  30.      0.   ]
 [  1.    126.     60.    ...   0.349  47.      1.   ]
 [  1.     93.     70.    ...   0.315  23.      0.   ]]


In [8]:
required_cols = ['Age','Gender','BMI']
bio_df = df.loc[:, df.columns.intersection(required_cols)]

In [9]:
# Simulate Contact_Email column

df['Contact_Email'] = df.index.map(lambda x: f"patient{x}@hospital.com")

In [10]:
print(df.columns.tolist())

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome', 'Contact_Email']


In [11]:
# Extract Biological Identity DataFrame

bio_df = df[['Age', 'BMI', 'Outcome']].copy()


# Extract Communication Channel as Series

email_series = df['Contact_Email']

In [12]:
# Object Types

print(type(bio_df))
print(type(email_series))

print("Bio DF Shape:", bio_df.shape)
print("Email Series Shape:", email_series.shape)

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
Bio DF Shape: (768, 3)
Email Series Shape: (768,)


In [13]:
# Summary Table

summary = pd.DataFrame({
    "Object": ["Biological Identity DF", "Email Series"],
    "Type": [type(bio_df), type(email_series)],
    "Shape": [bio_df.shape, email_series.shape],
    "Index": [bio_df.index, email_series.index]
})

summary

,Object,Type,Shape,Index
0,Biological Identity DF,<class 'pandas.DataFrame'>,"(768, 3)","RangeIndex(start=0, stop=768, step=1)"
1,Email Series,<class 'pandas.Series'>,"(768,)","RangeIndex(start=0, stop=768, step=1)"


In [14]:
# Contact_Email (Dummy)

df["Contact_Email"] = ["Patient" + str(i) + "@gmail.com" for i in range(len(df))]

In [15]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Contact_Email
0,6,148,72,35,0,33.6,0.627,50,1,Patient0@gmail.com
1,1,85,66,29,0,26.6,0.351,31,0,Patient1@gmail.com
2,8,183,64,0,0,23.3,0.672,32,1,Patient2@gmail.com
3,1,89,66,23,94,28.1,0.167,21,0,Patient3@gmail.com
4,0,137,40,35,168,43.1,2.288,33,1,Patient4@gmail.com


In [16]:
# Memory Usage

print(bio_df.memory_usage(deep=True))
print(email_series.memory_usage(deep=True))

Index       132
Age        6144
BMI        6144
Outcome    6144
dtype: int64
55318


# Task 2: The Temporal Cartographer (Multi-Column Date Assembly)

In [17]:
import numpy as np
import pandas as pd 

In [20]:
# Simulated dataset

data = {
    'Year':[2019,2020,2020,2021,2024],
    'Month':[1,2,2,5,2],
    'Day':[10,29,15,20,28],
    'Magnitude':[4.5,6.1,5.3,4.8,6.5]
}

df = pd.DataFrame(data)

# Save & reload using parse_dates

csv_path = "earthquake_sim.csv"
df.to_csv(csv_path,index=False)

quakes = pd.read_csv(csv_path)
quakes ["Event_Date"] = pd.to_datetime(quakes[["Year","Month","Day"]])

print(quakes.info())

# Leap Year Events

leap_years = quakes[quakes['Event_Date'].dt.is_leap_year]
print(leap_years)

# Busiest Month

print(quakes['Event_Date'].dt.month.value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Year        5 non-null      int64         
 1   Month       5 non-null      int64         
 2   Day         5 non-null      int64         
 3   Magnitude   5 non-null      float64       
 4   Event_Date  5 non-null      datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3)
memory usage: 332.0 bytes
None
   Year  Month  Day  Magnitude Event_Date
1  2020      2   29        6.1 2020-02-29
2  2020      2   15        5.3 2020-02-15
4  2024      2   28        6.5 2024-02-28
Event_Date
2    3
1    1
5    1
Name: count, dtype: int64


# Task 3: The Continental Translator (Delimiter & Encoding Mastery)

In [21]:
sample = """Emp_ID;Name;Country;Job_Field;Salary
201;García;Spain;Finance;65000
202;Müller;Germany;Engineering;72000
203;Dubois;France;HR;58000"""

with open("european_payroll.csv","w",encoding="latin-1") as f:
    f.write(sample)

payroll = pd.read_csv("european_payroll.csv",
                       sep=';',encoding='latin-1')

print(payroll)
print(payroll.describe(include='object'))
print(payroll['Job_Field'].value_counts())

   Emp_ID    Name  Country    Job_Field  Salary
0     201  García    Spain      Finance   65000
1     202  Müller  Germany  Engineering   72000
2     203  Dubois   France           HR   58000
          Name Country Job_Field
count        3       3         3
unique       3       3         3
top     García   Spain   Finance
freq         1       1         1
Job_Field
Finance        1
Engineering    1
HR             1
Name: count, dtype: int64


C:\Users\Darshan\AppData\Local\Temp\ipykernel_34996\3951557591.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(payroll.describe(include='object'))


# Task 4: The Memory Strategist (Deep RAM Diagnostics)

In [22]:
stock = pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/Auto.csv")

In [23]:
stock

,mpg,cylinders,displacement,horsepower,weight,acceleration,year,origin,name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino
...,...,...,...,...,...,...,...,...,...
387,27.0,4,140.0,86,2790,15.6,82,1,ford mustang gl
388,44.0,4,97.0,52,2130,24.6,82,2,vw pickup
389,32.0,4,135.0,84,2295,11.6,82,1,dodge rampage
390,28.0,4,120.0,79,2625,18.6,82,1,ford ranger


In [24]:
import numpy as np

# Simulate low-cardinality columns

stock['Exchange'] = np.tile(['NYSE','NASDAQ'], len(stock))[:len(stock)]
stock['Trade_Type'] = np.tile(['BUY','SELL'], len(stock))[:len(stock)]

In [25]:
print(stock.info(memory_usage='deep'))

<class 'pandas.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           392 non-null    float64
 1   cylinders     392 non-null    int64  
 2   displacement  392 non-null    float64
 3   horsepower    392 non-null    int64  
 4   weight        392 non-null    int64  
 5   acceleration  392 non-null    float64
 6   year          392 non-null    int64  
 7   origin        392 non-null    int64  
 8   name          392 non-null    str    
 9   Exchange      392 non-null    str    
 10  Trade_Type    392 non-null    str    
dtypes: float64(3), int64(5), str(3)
memory usage: 90.3 KB
None


In [26]:
before = stock.memory_usage(deep=True).sum()

In [27]:
stock['Exchange'] = stock['Exchange'].astype('category')
stock['Trade_Type'] = stock['Trade_Type'].astype('category')

In [28]:
after = stock.memory_usage(deep=True).sum()

In [29]:
print("Before:",before)
print("After:",after)
print(stock.describe())

Before: 92496
After: 51745
              mpg   cylinders  displacement  horsepower       weight  \
count  392.000000  392.000000    392.000000  392.000000   392.000000   
mean    23.445918    5.471939    194.411990  104.469388  2977.584184   
std      7.805007    1.705783    104.644004   38.491160   849.402560   
min      9.000000    3.000000     68.000000   46.000000  1613.000000   
25%     17.000000    4.000000    105.000000   75.000000  2225.250000   
50%     22.750000    4.000000    151.000000   93.500000  2803.500000   
75%     29.000000    8.000000    275.750000  126.000000  3614.750000   
max     46.600000    8.000000    455.000000  230.000000  5140.000000   

       acceleration        year      origin  
count    392.000000  392.000000  392.000000  
mean      15.541327   75.979592    1.576531  
std        2.758864    3.683737    0.805518  
min        8.000000   70.000000    1.000000  
25%       13.775000   73.000000    1.000000  
50%       15.500000   76.000000    1.000000  
75

# Task 5: The Precision Extractor (Selective Ingestion from Excel)

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_excel("Pharmaceutical_sales_ data.xlsx")
df

,Field,Description
0,Distributor,Name of Wholesaler
1,Customer Name,Name of customer
2,City,Customer's city
3,Country,Customer's country
4,Latitude,Customer's Geo Latitude
5,Longitude,Customer's Geo Longitude
6,Channel,"Class of buyer (Hospital, Pharmacy)"
7,Sub-channel,"Sector of buyer (Government, Private etc)"
8,Product Name,Name of Drug
9,Product Class,"Class of Drug (Antibiotics, etc)"


In [ ]:
xls = pd.ExcelFile("Pharmaceutical_sales_ data.xlsx")
print(xls.sheet_names)

['Fields Descriptions', 'Data']


In [8]:
temp = pd.read_excel("Pharmaceutical_sales_ data.xlsx",sheet_name = 0)
print(temp.columns.tolist())

['Field', 'Description']


In [10]:
temp = pd.read_excel("Pharmaceutical_sales_ data.xlsx", sheet_name="Data")
print(temp.columns.tolist())

['Distributor', 'Customer Name', 'City', 'Country', 'Latitude', 'Longitude', 'Channel', 'Sub-channel', 'Product Name', 'Product Class', 'Quantity', 'Price', 'Sales', 'Month', 'Year', 'Name of Sales Rep', 'Manager', 'Sales Team']


In [11]:
df = pd.read_excel("Pharmaceutical_sales_ data.xlsx", sheet_name="Data", usecols=["Product Name","Quantity","Sales","Month","Year"])
df

,Product Name,Quantity,Sales,Month,Year
0,Topipizole,4.0,1472.0,January,2018
1,Choriotrisin,7.0,4137.0,January,2018
2,Acantaine,30.0,1980.0,January,2018
3,Lioletine Refliruvax,6.0,2610.0,January,2018
4,Oxymotroban Fexoformin,20.0,9160.0,January,2018
...,...,...,...,...,...
254077,Pentastrin,919.0,456743.0,December,2020
254078,Abranatal Lysoprosate,432.0,294192.0,December,2020
254079,Adideine,320.0,216960.0,December,2020
254080,Feruprazole,565.0,64975.0,December,2020


In [12]:
df["InvoiceDate"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01")
df = df[["InvoiceDate","Product Name","Quantity","Sales"]]
df

,InvoiceDate,Product Name,Quantity,Sales
0,2018-01-01,Topipizole,4.0,1472.0
1,2018-01-01,Choriotrisin,7.0,4137.0
2,2018-01-01,Acantaine,30.0,1980.0
3,2018-01-01,Lioletine Refliruvax,6.0,2610.0
4,2018-01-01,Oxymotroban Fexoformin,20.0,9160.0
...,...,...,...,...
254077,2020-12-01,Pentastrin,919.0,456743.0
254078,2020-12-01,Abranatal Lysoprosate,432.0,294192.0
254079,2020-12-01,Adideine,320.0,216960.0
254080,2020-12-01,Feruprazole,565.0,64975.0


In [13]:
df.columns = ["InvoiceDate","ProductName","UnitsSold","Revenue"]
df

,InvoiceDate,ProductName,UnitsSold,Revenue
0,2018-01-01,Topipizole,4.0,1472.0
1,2018-01-01,Choriotrisin,7.0,4137.0
2,2018-01-01,Acantaine,30.0,1980.0
3,2018-01-01,Lioletine Refliruvax,6.0,2610.0
4,2018-01-01,Oxymotroban Fexoformin,20.0,9160.0
...,...,...,...,...
254077,2020-12-01,Pentastrin,919.0,456743.0
254078,2020-12-01,Abranatal Lysoprosate,432.0,294192.0
254079,2020-12-01,Adideine,320.0,216960.0
254080,2020-12-01,Feruprazole,565.0,64975.0


In [14]:
df.info()
df.tail()

<class 'pandas.DataFrame'>
RangeIndex: 254082 entries, 0 to 254081
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceDate  254082 non-null  datetime64[us]
 1   ProductName  254082 non-null  str           
 2   UnitsSold    254082 non-null  float64       
 3   Revenue      254082 non-null  float64       
dtypes: datetime64[us](1), float64(2), str(1)
memory usage: 7.8 MB


,InvoiceDate,ProductName,UnitsSold,Revenue
254077,2020-12-01,Pentastrin,919.0,456743.0
254078,2020-12-01,Abranatal Lysoprosate,432.0,294192.0
254079,2020-12-01,Adideine,320.0,216960.0
254080,2020-12-01,Feruprazole,565.0,64975.0
254081,2020-12-01,Feruprazole,1080.0,124200.0
